# Alpha101 Adaptive Research Factory

Notebook-first workflow for all 101 Kakushadze Formulaic Alphas on the cached India equity universes. The goal is not to force every alpha into the Alpha#1 repair path; it is to classify formula computability, input quality, family behavior, transform compatibility, decay, and portfolio usefulness versus the correct active equal-weight benchmark.

Primary formula source: Kakushadze, *101 Formulaic Alphas*, arXiv:1601.00991.

## Run Controls

`RUN_ALPHA101_REFRESH=False` is the normal mode. It loads existing artifacts if present, otherwise it resumes from per-alpha task caches and computes only what is missing. Set it to `True` only when you intentionally want to rebuild all task caches.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from research.alpha101_engine import ALPHA101_ARTIFACT_DIR, load_panel
from research.alpha101_factory import run_alpha101_factory
from research.alpha101_robustness import run_alpha101_robustness, run_alpha101_robustness_batch2

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RUN_ALPHA101_REFRESH = False
ALPHA101_REAGGREGATE = False
RUN_ALPHA101_ROBUSTNESS_REFRESH = False
RUN_ALPHA101_ROBUSTNESS_BATCH2_REFRESH = False
ALPHA101_MAX_WORKERS = 4
ARTIFACT_DIR = ALPHA101_ARTIFACT_DIR
ARTIFACT_DIR

## Data And Universe Audit

The factory uses cached real OHLCV data only. The close used by formulas is adjusted close; open/high/low are adjusted by the same close adjustment factor so mixed OHLC formulas are not distorted by corporate actions. VWAP is a labeled proxy: `(adjusted_high + adjusted_low + adjusted_close) / 3`.

In [ ]:
panel_rows = []
for panel_name in ["nifty500", "expanded"]:
    panel = load_panel(panel_name)
    panel_rows.append({
        "panel": panel_name,
        "start": panel.close.index.min().date(),
        "end": panel.close.index.max().date(),
        "sessions": len(panel.close),
        "symbols": panel.close.shape[1],
        "median_active_names": panel.active_mask.sum(axis=1).median(),
        "median_high_vol_names": panel.high_vol_mask.sum(axis=1).median(),
        "pit_risk": panel.pit_risk,
    })
display(pd.DataFrame(panel_rows))

## Run Or Load Factory

This single call produces every artifact: registry, input-quality report, operator validation, formula validation, IC panels, transform grid, decay report, portfolio report, leaderboard, shortlist, and final markdown report.

In [ ]:
outputs = run_alpha101_factory(
    max_workers=ALPHA101_MAX_WORKERS,
    refresh=RUN_ALPHA101_REFRESH,
    progress=True,
    reaggregate=ALPHA101_REAGGREGATE,
)
{k: v.shape for k, v in outputs.items()}

## Formula Registry Audit

In [ ]:
registry = outputs["registry"]
display(registry.head(20))
display(registry["input_quality_tier"].value_counts(dropna=False).rename("alphas").to_frame())
display(registry["family"].value_counts(dropna=False).rename("alphas").to_frame())

## Input Quality Audit

`exact_ohlcv` alphas use cached adjusted OHLCV directly. `proxy_vwap` alphas use typical price as VWAP proxy. `snapshot_industry` alphas use current constituent industry metadata and are explicitly not PIT industry backtests. `missing_cap` is untestable with the current cache.

In [ ]:
input_quality = outputs["input_quality"]
display(input_quality.groupby("input_quality_tier", dropna=False).size().rename("alphas").reset_index())
display(input_quality[input_quality["input_quality_tier"].str.contains("missing", na=False)])

## Operator Validation

In [ ]:
operator_validation = outputs["operator_validation"]
display(operator_validation)
assert operator_validation["passed"].all(), "Operator validation failed"

## Formula Validation

Every formula must either compute on each panel or be explicitly marked untestable/failed. This is where formula, missing input, and numerical failures are separated before judging signal quality.

In [ ]:
formula_validation = outputs["formula_validation"]
display(formula_validation.groupby(["panel", "computed"], dropna=False).size().rename("count").reset_index())
display(formula_validation[~formula_validation["computed"].fillna(False)].head(50))
display(formula_validation.sort_values("non_null_scores", ascending=False).head(20))

## Family And Transform Compatibility

Transform selection is reported separately from baseline metrics to reduce data-mining fog. Family rules determine what gets tried; they do not declare a transform valid by itself.

In [ ]:
display(outputs["family_classification"].head(30))
display(outputs["transform_compatibility"])

## IC And Horizon Diagnostics

This table ranks formula/transform/horizon combinations by mean rank IC. Use it to answer: is the formula directionally informative before portfolio construction?

In [ ]:
metric_panel = outputs["metric_panel"]
ic_cols = ["panel", "alpha_id", "family", "transform", "horizon_days", "mean_rank_ic", "rank_icir", "positive_ic_rate", "observations"]
display(metric_panel.sort_values("mean_rank_ic", ascending=False)[ic_cols].head(40))
display(metric_panel.sort_values("mean_rank_ic", ascending=True)[ic_cols].head(20))

## Decay Diagnostics

The decay report compares IC by era and includes a `recent_minus_old` row. This is not a final verdict by itself, but it flags families that appear to have weakened in the latest period.

In [ ]:
decay_report = outputs["decay_report"]
display(decay_report[decay_report["era"].eq("recent_minus_old")].sort_values("mean_rank_ic").head(30))
display(decay_report.pivot_table(index="family", columns="era", values="mean_rank_ic", aggfunc="median"))

## Portfolio Reality Check

The primary portfolio metric is active return versus the equal-weight active universe after costs. The report keeps alpha return, benchmark return, excess return, turnover, drawdown, Sharpe, Sortino, and hit rate side by side.

In [ ]:
portfolio_report = outputs["portfolio_report"]
portfolio_cols = [
    "panel", "alpha_id", "family", "signal_transform", "mask", "strategy", "cost_bps",
    "alpha_cagr", "benchmark_cagr", "active_cagr", "active_sharpe", "alpha_avg_daily_turnover", "active_max_drawdown", "avg_names",
]
display(portfolio_report.query("cost_bps == 20.0").sort_values("active_sharpe", ascending=False)[portfolio_cols].head(50))
display(portfolio_report.query("cost_bps == 20.0").pivot_table(index="family", columns="strategy", values="active_sharpe", aggfunc="median"))

## Transform Grid Summary

This aggregates transform/portfolio templates across alphas. Treat it as a research-map of where signal extraction tends to survive costs, not as a tradable model selection step.

In [ ]:
transform_grid = outputs["transform_grid"]
display(transform_grid.sort_values("mean_active_sharpe", ascending=False).head(60))

## Leaderboard And Candidate Shortlist

The leaderboard combines IC, portfolio active Sharpe after 20 bps, turnover, and recent decay. The shortlist is deliberately research-oriented: candidates and feature-only alphas are kept separate from tradable approval.

In [ ]:
leaderboard = outputs["leaderboard"]
shortlist = outputs["shortlist"]
leader_cols = [
    "panel", "alpha_id", "family", "input_quality_tier", "classification", "best_5d_ic", "rank_icir",
    "positive_ic_rate", "best_20bps_active_sharpe", "best_signal_transform", "best_strategy", "best_mask", "recent_minus_old_ic", "research_score",
]
display(leaderboard[leader_cols].head(60))
display(shortlist[leader_cols].head(60))
display(leaderboard.groupby(["panel", "classification"], dropna=False).size().rename("count").reset_index())

## Candidate Robustness And Data-Risk Triage

The factory leaderboard is discovery only. This stage reruns only the top candidates, selects transform/portfolio/mask combinations on train windows, scores test windows versus the equal-weight active universe, and separates exact-OHLCV, proxy-VWAP, snapshot-industry, and Alpha#1 baseline lanes.

In [ ]:
robustness = run_alpha101_robustness(
    refresh=RUN_ALPHA101_ROBUSTNESS_REFRESH,
    clean_n=12,
    proxy_n=8,
    snapshot_n=8,
    progress=True,
)
{k: v.shape for k, v in robustness.items()}

## Robustness Validation

In [ ]:
display(robustness["validation"])
assert robustness["validation"]["passed"].all(), "Robustness validation failed"

## Robustness Shortlist

`promote_to_deeper_research` survived walk-forward active-return checks after train-only selection. `feature_only` retained signal information but did not clear the portfolio hurdle. `proxy_dependent` is rejected until better VWAP data exists. `snapshot_metadata_risk` is research-only until point-in-time industry metadata exists.

In [ ]:
robust_cols = [
    "panel", "alpha_id", "robustness_lane", "input_quality_tier", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "worst_test_drawdown",
    "best_mask", "best_signal_transform", "best_strategy",
]
display(robustness["shortlist"].sort_values("median_test_active_sharpe", ascending=False)[robust_cols].head(60))
display(robustness["shortlist"].groupby(["robustness_lane", "final_status"], dropna=False).size().rename("count").reset_index())

## Cost And Universe Sensitivity

In [ ]:
display(robustness["cost_sensitivity"].pivot_table(
    index=["panel", "alpha_id", "robustness_lane"],
    columns="cost_bps",
    values="test_active_sharpe",
    aggfunc="median",
).sort_values(20.0, ascending=False).head(40))

display(robustness["universe_sensitivity"].pivot_table(
    index=["panel", "alpha_id", "robustness_lane"],
    columns="selected_mask",
    values="test_active_sharpe",
    aggfunc="median",
).sort_values("high_vol_top100", ascending=False).head(40))

## Proxy And Snapshot Risk

In [ ]:
display(robustness["proxy_sensitivity"].sort_values(["proxy_dependent", "sharpe_range"], ascending=[False, False]).head(80))
display(robustness["industry_snapshot_risk"].sort_values("snapshot_minus_identity_sharpe", ascending=False).head(40))

## Preprocessing And Scaling Caveats

The current Alpha101 factory is acceptable as a **candidate discovery and robustness triage layer**, but preprocessing is not yet production-clean. The scaling and transform framework is mostly correct: OHLC fields are adjusted with `adj_close / close`, cross-sectional ranks/z-scores are computed by date inside active masks, `scale()` uses row-wise L1 normalization, orientation is causal, and robustness selection is train-only before OOS scoring.

The remaining preprocessing risks are important before capital promotion:

- Several formula implementations replace invalid rolling values with `0` or `1`; this can convert missing warmup data into signal. A stricter NaN-preserving formula pass is needed.
- Liquidity notional currently uses adjusted close times volume; raw close times volume is better for traded rupee volume and capacity checks.
- VWAP remains a proxy, so proxy-VWAP alphas need real VWAP before promotion. Current proxy sensitivity demotes unstable names, but it does not prove proxy names are fully clean.
- Industry metadata is a current snapshot, not point-in-time; industry-neutral alphas remain research-only under `snapshot_metadata_risk`.
- Current constituent universes carry survivorship/PIT-membership risk.
- Add explicit selected-name forward-return, stale-price, warmup-NaN, and invalid-return audits before treating promoted candidates as tradable.

Practical conclusion: **scaling is mostly fine; preprocessing/data-validity hygiene is the next correctness gate.** Rerun promoted exact-OHLCV candidates after a strict NaN and liquidity-notional audit.


## Robustness Final Report

In [ ]:
robust_report_path = ARTIFACT_DIR / "alpha101_robustness_final_report.md"
if robust_report_path.exists():
    display(Markdown(robust_report_path.read_text()))
else:
    display(Markdown("Robustness report has not been written yet. Run the robustness cell above."))

## Batch 2 Clean Near-Miss Robustness

This reruns only the 20 clean exact-OHLCV near-miss candidates from discovery. It uses the same train-only walk-forward selection, active benchmark comparison, universe sensitivity, and cost grid as Batch 1, while excluding proxy, snapshot-industry, decayed, and discarded alphas.

In [ ]:
batch2 = run_alpha101_robustness_batch2(
    refresh=RUN_ALPHA101_ROBUSTNESS_BATCH2_REFRESH,
    progress=True,
)
{k: v.shape for k, v in batch2.items()}

## Batch 2 Validation

In [ ]:
display(batch2["validation"])
assert batch2["validation"]["passed"].all(), "Batch 2 robustness validation failed"

## Batch 2 Shortlist

Batch 2 uses the same final status rules as Batch 1: `promote_to_deeper_research`, `feature_only`, or `discard`. These are still research classifications, not capital allocation approvals.

In [ ]:
batch2_cols = [
    "panel", "alpha_id", "robustness_lane", "input_quality_tier", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "worst_test_drawdown",
    "best_mask", "best_signal_transform", "best_strategy",
]
display(batch2["shortlist"].sort_values("median_test_active_sharpe", ascending=False)[batch2_cols])
display(batch2["shortlist"].groupby("final_status", dropna=False).size().rename("count").reset_index())

## Combined Exact-OHLCV Promotions

This table keeps Batch 1 and Batch 2 separate, then filters to promoted exact-OHLCV candidates so the clean research lane is visible without proxy or snapshot metadata risk mixed in.

In [ ]:
combined_promoted_exact = batch2["combined_shortlist"].loc[
    batch2["combined_shortlist"]["final_status"].eq("promote_to_deeper_research")
    & batch2["combined_shortlist"].get("input_quality_tier", pd.Series(index=batch2["combined_shortlist"].index, dtype=object)).eq("exact_ohlcv")
].sort_values(["batch", "median_test_active_sharpe"], ascending=[True, False])

combined_cols = [
    "batch", "panel", "alpha_id", "robustness_lane", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "best_mask", "best_signal_transform", "best_strategy",
]
display(combined_promoted_exact[combined_cols])
display(combined_promoted_exact.groupby("batch", dropna=False).size().rename("promoted_exact_ohlcv_count").reset_index())

## Batch 2 Final Report

In [ ]:
batch2_report_path = ARTIFACT_DIR / "alpha101_robustness_batch2_final_report.md"
if batch2_report_path.exists():
    display(Markdown(batch2_report_path.read_text()))
else:
    display(Markdown("Batch 2 robustness report has not been written yet. Run the Batch 2 cell above."))

## Final Report

In [ ]:
report_path = ARTIFACT_DIR / "alpha101_final_report.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    display(Markdown("Final report has not been written yet. Run the factory cell above."))

## Artifact Manifest

In [ ]:
artifact_names = [
    "alpha101_formula_registry.csv",
    "alpha101_input_quality_report.csv",
    "alpha101_operator_validation.csv",
    "alpha101_formula_validation.csv",
    "alpha101_family_classification.csv",
    "alpha101_transform_compatibility.csv",
    "alpha101_metric_panel.csv",
    "alpha101_transform_grid.csv",
    "alpha101_decay_report.csv",
    "alpha101_portfolio_report.csv",
    "alpha101_leaderboard.csv",
    "alpha101_candidate_shortlist.csv",
    "alpha101_final_report.md",
    "alpha101_robustness_final_report.md",
    "alpha101_robustness_shortlist.csv",
    "alpha101_robustness_validation.csv",
    "alpha101_industry_snapshot_risk_report.csv",
    "alpha101_proxy_sensitivity_report.csv",
    "alpha101_robustness_universe_sensitivity.csv",
    "alpha101_robustness_cost_sensitivity.csv",
    "alpha101_robustness_walk_forward.csv",
    "alpha101_robustness_candidate_lanes.csv",
    "alpha101_robustness_batch2_final_report.md",
    "alpha101_robustness_batch2_shortlist.csv",
    "alpha101_robustness_batch2_validation.csv",
    "alpha101_robustness_batch2_universe_sensitivity.csv",
    "alpha101_robustness_batch2_cost_sensitivity.csv",
    "alpha101_robustness_batch2_walk_forward.csv",
    "alpha101_robustness_batch2_candidate_lanes.csv",
    "alpha101_robustness_combined_shortlist.csv",
    "alpha101_valid_alpha_performance_metrics.csv",
    "alpha101_investor_carry_report.md",
    "alpha101_promoted_ensemble_carry_metrics.csv",
    "alpha101_valid_alpha_carry_metrics.csv",
    "alpha101_valid_alpha_carry_fold_metrics.csv",
]
manifest = pd.DataFrame({
    "artifact": artifact_names,
    "path": [str(ARTIFACT_DIR / name) for name in artifact_names],
    "exists": [(ARTIFACT_DIR / name).exists() for name in artifact_names],
})
display(manifest)